# Perseus MCP Advanced Search: Tutorial and Parameter Reference

## Table of contents (ToC) <a class="anchor" id="TOC"></a>
* <a href="#introduction">1 - Purpose and place in the tutorial series</a>
* <a href="#architecture">2 - Understand the search architecture</a>
* <a href="#parameter-reference">3 - Complete `search_perseus` parameter reference</a>
* <a href="#setup">4 - Install dependencies and load the server</a>
* <a href="#helpers">5 - Build reusable result helpers</a>
* <a href="#live-schema">6 - Inspect and verify the live tool schema</a>
* <a href="#normalization">7 - Understand Unicode, Beta Code, and NFC normalization</a>
* <a href="#language">8 - Understand what `language` does and does not do</a>
* <a href="#discover-scope">9 - Discover stable author and work scopes</a>
* <a href="#form-lemma">10 - Compare surface-form and lemma search</a>
* <a href="#response-anatomy">11 - Read the anatomy of a search response</a>
* <a href="#result-formats">12 - Compare `instances` and `passages` result formats</a>
* <a href="#operators">13 - Preserve quoted, exclusion, OR, wildcard, and fuzzy syntax</a>
* <a href="#operator-matrix">14 - Run a compact operator experiment</a>
* <a href="#pagination">15 - Navigate result pages safely</a>
* <a href="#explicit-scopes">16 - Use server-side textgroup and work scopes</a>
* <a href="#author-scope">17 - Understand author resolution and fallback filtering</a>
* <a href="#recipes">18 - Query recipes and decision guide</a>
* <a href="#preflight">19 - Validate search arguments before calling</a>
* <a href="#workflow">20 - Build an end-to-end evidence packet</a>
* <a href="#interpretation">21 - Interpret counts, snippets, and URNs cautiously</a>
* <a href="#operations">22 - Network, performance, and reproducibility</a>
* <a href="#troubleshooting">23 - Troubleshooting reference</a>
* <a href="#next-steps">24 - Continue learning</a>
* <a href="#sources">25 - Sources</a>
* <a href="#required-libraries">26 - Required libraries</a>
* <a href="#notebook-version">27 - Notebook version</a>

## 1 - Purpose and place in the tutorial series <a class="anchor" id="introduction"></a>
##### [Back to ToC](#TOC)

This notebook is a focused **tutorial and reference handbook** for the `search_perseus` MCP tool. It moves beyond a first search and explains how every search argument changes normalization, scope, paging, or upstream response behavior.

The preceding notebooks establish the foundations:

- notebook `01_` introduces Perseus CTS and CTS URNs;
- notebook `02_` introduces direct Scaife search and CTS navigation;
- notebook `03_` introduces the FastMCP client workflow;
- notebook `04_` teaches a basic Greek search-to-navigation workflow;
- notebook `05_` catalogs the complete MCP tool surface;
- notebook `06_` lets an OpenRouter-hosted model select and call MCP tools.

This notebook concentrates on advanced library search. By the end, you should be able to:

1. choose between surface-form and lemma search;
2. predict how Unicode Greek, Beta Code, and ambiguous ASCII input are normalized;
3. preserve Scaife operator characters without accidental Beta Code conversion;
4. use pagination and `instances`/`passages` response formats;
5. scope a search explicitly by CTS textgroup or work URN;
6. understand when `author` becomes a server-side scope and when it falls back to current-page filtering;
7. extract compact, auditable evidence rows from live JSON results;
8. record enough search provenance to reproduce or critique a result later.

> The examples use live Perseus and Scaife services. Counts, ordering, indexed editions, snippets, and response details may change. Assertions in this notebook check contracts and relationships rather than fixed result totals.

## 2 - Understand the search architecture <a class="anchor" id="architecture"></a>
##### [Back to ToC](#TOC)

`search_perseus` is an MCP wrapper around Scaife's JSON library-search endpoint, with optional help from the Perseus CTS inventory.

```text
Notebook / external MCP client
        |
        | call_tool("search_perseus", arguments)
        v
Local Perseus MCP server
        |-- validate enums and page number
        |-- normalize Unicode Greek or Beta Code
        |-- optionally resolve author via cached CTS capabilities
        |-- construct Scaife search parameters
        v
Scaife JSON search endpoint
        |
        v
JSON text returned through MCP
        |
        | optional local author fallback filter
        v
Client parses JSON and interprets results
```

Three boundaries matter:

| Boundary | Consequence |
|---|---|
| MCP versus Scaife | MCP validates and normalizes arguments, but Scaife determines index behavior, ranking, snippets, and operator semantics |
| CTS versus Scaife | CTS discovery supplies stable author/work identity; search hits often use Scaife edition URNs that differ from Perseus CTS editions |
| Upstream JSON versus client interpretation | A successful HTTP/MCP call does not prove that a query means what the researcher intended; inspect the normalized query, scope metadata, and hits |

This notebook focuses on `search_perseus`, which searches Scaife's library index. Notebook `08_` covers the separate reader-search tools `search_within_text` and `get_passage_highlights`.

## 3 - Complete `search_perseus` parameter reference <a class="anchor" id="parameter-reference"></a>
##### [Back to ToC](#TOC)

Current public signature:

```python
search_perseus(
    query,
    language="greek",
    query_format="auto",
    author=None,
    search_kind="form",
    preserve_operators=False,
    page_num=1,
    text_group=None,
    work=None,
    result_format="instances",
)
```

| Argument | Default | Accepted values / shape | Effect |
|---|---|---|---|
| `query` | required | non-empty string in practice | Search expression sent to Scaife after normalization |
| `language` | `"greek"` | names/codes such as `greek`, `grc`, `latin`, `lat` | Controls query normalization and author-resolution language; it is **not** sent as a Scaife corpus-language filter |
| `query_format` | `"auto"` | `auto`, `unicode`, `betacode` | Controls Greek query conversion unless operators are being preserved |
| `author` | `None` | author name, partial name, or CTS textgroup URN | Resolves against CTS; may add a server-side textgroup scope or trigger local current-page filtering |
| `search_kind` | `"form"` | `form`, `lemma` | Selects visible surface-form search or indexed lemma search |
| `preserve_operators` | `False` | boolean | Bypasses Beta Code conversion and preserves operator syntax, while NFC-normalizing the whole query |
| `page_num` | `1` | integer ≥ 1 | Selects a Scaife library-search page |
| `text_group` | `None` | CTS textgroup URN | Server-side Scaife scope, normally an author/textgroup such as `urn:cts:greekLit:tlg0012` |
| `work` | `None` | CTS work URN | Server-side Scaife scope, such as `urn:cts:greekLit:tlg0012.tlg001` |
| `result_format` | `"instances"` | `instances`, `passages` | Requests the upstream result representation |

The return value is JSON serialized as MCP text content. Parse it with `json.loads(tool_text(result))` before inspecting fields.

## 4 - Install dependencies and load the server <a class="anchor" id="setup"></a>
##### [Back to ToC](#TOC)

The setup follows notebooks `03_`–`06_`:

- install the declared notebook dependencies;
- locate the repository root from the current working directory;
- pin the metadata cache to the repository-level `.cache/perseus-mcp` directory;
- import and reload `perseus_mcp.server` so local edits are visible;
- obtain the in-process FastMCP server object.

Importing `perseus_mcp.server` does not launch a separate command-line server. `Client(mcp)` connects directly to the loaded FastMCP object.

In [1]:
%pip install --quiet "fastmcp>=2.12.0" "httpx>=0.27.0" "python-dotenv>=1.0.0"

Note: you may need to restart the kernel to use updated packages.


In [2]:
from datetime import datetime, timezone
from pathlib import Path
import html
import importlib
import json
import os
import re
import sys
import unicodedata

from IPython.display import Markdown, display

START = Path.cwd().resolve()
for candidate in [START, *START.parents]:
    if (candidate / "src" / "perseus_mcp" / "server.py").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError(
        f"Could not find src/perseus_mcp/server.py from {START}. Open this notebook inside the Perseus-mcp repository."
    )

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.environ.setdefault(
    "PERSEUS_MCP_CACHE_DIR",
    str(REPO_ROOT / ".cache" / "perseus-mcp"),
)

from fastmcp import Client
from perseus_mcp import server

server = importlib.reload(server)
mcp = server.mcp

print(f"Repository root: {REPO_ROOT}")
print(f"Cache directory: {os.environ['PERSEUS_MCP_CACHE_DIR']}")
print(f"Loaded MCP server: {mcp.name}")

Repository root: D:\Onedrive\GitHub\Perseus-mcp
Cache directory: D:\Onedrive\GitHub\Perseus-mcp\.cache\perseus-mcp
Loaded MCP server: perseus


## 5 - Build reusable result helpers <a class="anchor" id="helpers"></a>
##### [Back to ToC](#TOC)

Search responses can contain long HTML-highlighted snippets and nested passage/text metadata. These helpers:

- extract MCP text blocks;
- parse serialized JSON;
- remove highlight tags for compact console display;
- tolerate both nested and more direct result shapes;
- preserve the passage URN, edition/work labels, citation, and snippet together;
- summarize pagination and author-scope metadata without assuming fixed counts.

The raw JSON remains the authoritative response. Compact rows are a viewing aid, not a replacement for fields needed by a particular research project.

In [3]:
TAG_RE = re.compile(r"<[^>]+>")


def tool_text(result):
    return "\n".join(
        block.text
        for block in result.content
        if getattr(block, "text", None) is not None
    )


async def call_json(client, tool_name, arguments=None):
    result = await client.call_tool(tool_name, arguments or {})
    return json.loads(tool_text(result))


def clean_snippet(value):
    if isinstance(value, list):
        value = " ".join(str(item) for item in value)
    return " ".join(html.unescape(TAG_RE.sub("", value or "")).split())


def result_passage(result):
    passage = result.get("passage")
    return passage if isinstance(passage, dict) else result


def passage_labels(passage):
    text = passage.get("text") or {}
    labels = [
        ancestor.get("label")
        for ancestor in text.get("ancestors", []) or []
        if ancestor.get("label")
    ]
    if text.get("label"):
        labels.append(text["label"])
    return labels


def compact_search_rows(data, limit=5):
    rows = []
    for result in (data.get("results") or [])[:limit]:
        passage = result_passage(result)
        text = passage.get("text") or result.get("text") or {}
        content = result.get("content") or passage.get("content") or []
        rows.append(
            {
                "passage_urn": passage.get("urn") or result.get("urn"),
                "citation": passage.get("citation") or result.get("citation"),
                "text_urn": text.get("urn"),
                "labels": passage_labels(passage),
                "snippet": clean_snippet(content),
            }
        )
    return rows


def summarize_search(data, row_limit=3):
    page = data.get("page") or {}
    return {
        "normalized_q": data.get("q"),
        "total_count": data.get("total_count"),
        "page_number": page.get("number"),
        "num_pages": page.get("num_pages"),
        "returned_results": len(data.get("results") or []),
        "author_scope": data.get("author_scope"),
        "rows": compact_search_rows(data, row_limit),
    }


def print_search_summary(label, data, row_limit=3):
    print(f"\n## {label}")
    print(json.dumps(summarize_search(data, row_limit), ensure_ascii=False, indent=2))

## 6 - Inspect and verify the live tool schema <a class="anchor" id="live-schema"></a>
##### [Back to ToC](#TOC)

Documentation can drift when a tool gains or loses arguments. `client.list_tools()` exposes the schema currently registered by the loaded server.

The cell below:

1. retrieves the live `search_perseus` definition;
2. prints its description and JSON Schema;
3. compares its property names with the parameter reference maintained above;
4. fails if the notebook and server no longer agree.

The schema is the machine-readable call contract. Semantic details—such as the fact that `language` is not a corpus filter—still require the description and implementation notes.

In [4]:
async with Client(mcp) as client:
    live_tools = await client.list_tools()

tool_by_name = {tool.name: tool for tool in live_tools}
search_tool = tool_by_name["search_perseus"]
live_search_arguments = set((search_tool.inputSchema or {}).get("properties", {}))
expected_search_arguments = {
    "query",
    "language",
    "query_format",
    "author",
    "search_kind",
    "preserve_operators",
    "page_num",
    "text_group",
    "work",
    "result_format",
}

schema_drift = {
    "missing_from_live_schema": sorted(expected_search_arguments - live_search_arguments),
    "undocumented_live_arguments": sorted(live_search_arguments - expected_search_arguments),
}

print(search_tool.description)
print("\nLive JSON Schema:")
print(json.dumps(search_tool.inputSchema, ensure_ascii=False, indent=2))
print("\nSchema drift check:")
print(json.dumps(schema_drift, indent=2))

assert not any(schema_drift.values()), "Notebook parameter reference has drifted from perseus_mcp.server"

Search Perseus texts via Scaife API.

For Greek searches, `query` may be Unicode Greek or Beta Code.  The default
`query_format="auto"` detects explicit Beta Code marks such as `=`, `/`,
`(`, `)`, and `*`, and also accepts short unaccented Beta Code queries such
as `logos`.  Set `query_format="betacode"` to force conversion or
`query_format="unicode"` to preserve ASCII text in Greek searches.
The `language` value determines whether Greek query normalization is applied;
it is not sent to Scaife as a corpus language filter.
Optional `author` resolves a CTS author/textgroup name or URN, then locally
filters the current Scaife result page to matching CTS URN prefixes.
`search_kind` may be "form" or "lemma". Set `preserve_operators=True` for
Scaife operator queries such as quoted phrases, `-`, `|`, `*`, or `~`.
Optional `page_num`, `text_group`, `work`, and `result_format` are passed
to Scaife's library search endpoint. When `author` resolves to exactly one
CTS textgroup and no explicit `te

## 7 - Understand Unicode, Beta Code, and NFC normalization <a class="anchor" id="normalization"></a>
##### [Back to ToC](#TOC)

For a Greek-language search, normalization follows this decision path:

```text
preserve_operators=True?
  yes -> NFC-normalize the complete query; do not perform Beta Code conversion
  no  -> query_format="unicode"?
           yes -> preserve characters and NFC-normalize
         query_format="betacode"?
           yes -> convert Beta Code to Greek, then NFC-normalize
         query_format="auto"?
           yes -> convert if explicit Beta Code markers are present,
                  or if the compact query is short ASCII letters only;
                  otherwise preserve and NFC-normalize
```

Important consequences:

- `mh=nin` with `betacode` becomes `μῆνιν`;
- `logos` with `auto` looks like short Beta Code and becomes `λογος`;
- `logos` with `unicode` remains ASCII `logos`;
- Beta Code `*` marks uppercase, while Scaife may interpret `*` as a wildcard;
- Beta Code `|` marks iota subscript, while an operator query may use `|` as OR-style syntax;
- with `preserve_operators=True`, `query_format` does not convert Beta Code—the query must already be in the desired character form.

The next cell calls private normalization helpers only as a local implementation diagnostic. External MCP clients should use the public `search_perseus` tool rather than depend on underscore-prefixed Python functions.

In [5]:
normalization_cases = [
    {
        "label": "Unicode Greek",
        "query": "μῆνιν",
        "language": "greek",
        "query_format": "unicode",
        "preserve_operators": False,
    },
    {
        "label": "forced Beta Code",
        "query": "mh=nin",
        "language": "greek",
        "query_format": "betacode",
        "preserve_operators": False,
    },
    {
        "label": "ambiguous ASCII with auto",
        "query": "logos",
        "language": "greek",
        "query_format": "auto",
        "preserve_operators": False,
    },
    {
        "label": "ambiguous ASCII forced Unicode",
        "query": "logos",
        "language": "greek",
        "query_format": "unicode",
        "preserve_operators": False,
    },
    {
        "label": "operator expression preserved",
        "query": "μῆνιν | ἄειδε",
        "language": "greek",
        "query_format": "auto",
        "preserve_operators": True,
    },
]

normalization_report = []
for case in normalization_cases:
    normalized = server._normalize_query_for_search(
        case["query"],
        case["language"],
        case["query_format"],
        case["preserve_operators"],
    )
    normalization_report.append(
        {
            **case,
            "normalized": normalized,
            "is_nfc": unicodedata.is_normalized("NFC", normalized),
        }
    )

print(json.dumps(normalization_report, ensure_ascii=False, indent=2))

assert normalization_report[0]["normalized"] == "μῆνιν"
assert normalization_report[1]["normalized"] == "μῆνιν"
assert normalization_report[2]["normalized"] == "λογος"
assert normalization_report[3]["normalized"] == "logos"
assert normalization_report[4]["normalized"] == "μῆνιν | ἄειδε"

[
  {
    "label": "Unicode Greek",
    "query": "μῆνιν",
    "language": "greek",
    "query_format": "unicode",
    "preserve_operators": false,
    "normalized": "μῆνιν",
    "is_nfc": true
  },
  {
    "label": "forced Beta Code",
    "query": "mh=nin",
    "language": "greek",
    "query_format": "betacode",
    "preserve_operators": false,
    "normalized": "μῆνιν",
    "is_nfc": true
  },
  {
    "label": "ambiguous ASCII with auto",
    "query": "logos",
    "language": "greek",
    "query_format": "auto",
    "preserve_operators": false,
    "normalized": "λογος",
    "is_nfc": true
  },
  {
    "label": "ambiguous ASCII forced Unicode",
    "query": "logos",
    "language": "greek",
    "query_format": "unicode",
    "preserve_operators": false,
    "normalized": "logos",
    "is_nfc": true
  },
  {
    "label": "operator expression preserved",
    "query": "μῆνιν | ἄειδε",
    "language": "greek",
    "query_format": "auto",
    "preserve_operators": true,
    "normalized": 

## 8 - Understand what `language` does and does not do <a class="anchor" id="language"></a>
##### [Back to ToC](#TOC)

The `language` argument has two local roles:

1. it determines whether Greek normalization is applied (`greek`, `grc`, and related forms normalize to Scaife code `gr`);
2. when `author` is supplied, it helps filter the CTS author/work inventory during author resolution.

It is **not** added to the Scaife library-search request as a corpus-language filter. Therefore:

- `language="greek"` does not by itself guarantee that every returned indexed resource is Greek;
- `language="latin"` prevents Greek/Beta Code conversion, but it does not independently scope the corpus to Latin;
- use a discovered `text_group` or `work` URN when corpus identity matters;
- inspect returned text metadata and URNs rather than inferring corpus language from the input argument.

This distinction is easy to miss because the same word—language—can describe input normalization, inventory filtering, work language, or corpus scope. In this tool, explicit CTS scope URNs are the reliable search boundary.

## 9 - Discover stable author and work scopes <a class="anchor" id="discover-scope"></a>
##### [Back to ToC](#TOC)

Server-side search scopes use CTS identity levels:

| Scope level | Example | Meaning |
|---|---|---|
| Textgroup | `urn:cts:greekLit:tlg0012` | Homer author/textgroup |
| Work | `urn:cts:greekLit:tlg0012.tlg001` | Homer's *Iliad* independent of edition |
| Edition | `urn:cts:greekLit:tlg0012.tlg001.perseus-grc2` | A particular Scaife/CTS resource; not accepted by the `work` field as the intended level |
| Passage | `...:1.1` | A cited location, not a library-search scope field here |

Do not guess these identifiers from memory. The cell below discovers the *Iliad*, selects the exact Homer match, and stores the textgroup and work URNs used throughout the notebook.

No edition URN is needed for `search_perseus` library scoping. The returned search hits will identify the indexed Scaife editions.

In [6]:
async with Client(mcp) as client:
    iliad_matches = await call_json(
        client,
        "get_work_resources",
        {"urn_or_title": "Iliad"},
    )

iliad_match = next(
    (
        match
        for match in iliad_matches.get("matches", [])
        if "Iliad" in match.get("work", {}).get("titles", [])
        and "Homer" in match.get("author", {}).get("names", [])
    ),
    None,
)
if iliad_match is None:
    raise RuntimeError("The current CTS inventory returned no exact Homer/Iliad match.")

HOMER_TEXTGROUP = iliad_match["author"]["urn"]
ILIAD_WORK = iliad_match["work"]["urn"]

print(
    json.dumps(
        {
            "author_names": iliad_match["author"].get("names"),
            "text_group": HOMER_TEXTGROUP,
            "work_titles": iliad_match["work"].get("titles"),
            "work": ILIAD_WORK,
            "advertised_editions": iliad_match["work"].get("editions"),
        },
        ensure_ascii=False,
        indent=2,
    )
)

{
  "author_names": [
    "Homer"
  ],
  "text_group": "urn:cts:greekLit:tlg0012",
  "work_titles": [
    "Iliad"
  ],
  "work": "urn:cts:greekLit:tlg0012.tlg001",
  "advertised_editions": [
    {
      "type": "edition",
      "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1",
      "label": "Iliad",
      "description": "Perseus:bib:oclc,29448041, Homer. Homeri Opera in five volumes. Oxford, Oxford University Press. 1920."
    }
  ]
}


## 10 - Compare surface-form and lemma search <a class="anchor" id="form-lemma"></a>
##### [Back to ToC](#TOC)

`search_kind="form"` searches the visible indexed token or expression. Use it when orthography or inflection matters—for example, the accusative singular `μῆνιν`.

`search_kind="lemma"` searches the lexical headword assigned by the index. Use it when you want indexed inflected forms associated with a lemma—for example, `μῆνις`.

| Research intention | Recommended kind |
|---|---|
| Find this exact written form | `form` |
| Compare spelling/accent variants explicitly | separate `form` searches |
| Find inflected forms indexed under one headword | `lemma` |
| Audit lemmatization quality | compare `form` and `lemma`, then inspect passages |

Lemma search depends on upstream morphological indexing. It is not a guarantee that every philologically plausible form is indexed under the expected headword. Form and lemma counts answer different questions and should not be compared as if one were simply a larger version of the other.

In [7]:
FORM_ARGUMENTS = {
    "query": "μῆνιν",
    "language": "greek",
    "query_format": "unicode",
    "search_kind": "form",
    "work": ILIAD_WORK,
    "result_format": "instances",
}
LEMMA_ARGUMENTS = {
    "query": "μῆνις",
    "language": "greek",
    "query_format": "unicode",
    "search_kind": "lemma",
    "work": ILIAD_WORK,
    "result_format": "instances",
}

async with Client(mcp) as client:
    form_search = await call_json(client, "search_perseus", FORM_ARGUMENTS)
    lemma_search = await call_json(client, "search_perseus", LEMMA_ARGUMENTS)

print_search_summary("Surface form μῆνιν in the Iliad", form_search)
print_search_summary("Lemma μῆνις in the Iliad", lemma_search)

print("\nComparison:")
print(
    json.dumps(
        {
            "form_total": form_search.get("total_count"),
            "lemma_total": lemma_search.get("total_count"),
            "form_normalized_q": form_search.get("q"),
            "lemma_normalized_q": lemma_search.get("q"),
        },
        ensure_ascii=False,
        indent=2,
    )
)


## Surface form μῆνιν in the Iliad
{
  "normalized_q": "μῆνιν",
  "total_count": 9,
  "page_number": 1,
  "num_pages": 1,
  "returned_results": 9,
  "author_scope": null,
  "rows": [
    {
      "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.75",
      "citation": null,
      "text_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
      "labels": [
        "Homer",
        "Iliad",
        "Ἰλιάς"
      ],
      "snippet": "μῆνιν Ἀπόλλωνος ἑκατηβελέταο ἄνακτος·"
    },
    {
      "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:5.444",
      "citation": null,
      "text_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
      "labels": [
        "Homer",
        "Iliad",
        "Ἰλιάς"
      ],
      "snippet": "μῆνιν ἀλευάμενος ἑκατηβόλου Ἀπόλλωνος."
    },
    {
      "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:16.711",
      "citation": null,
      "text_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
      "labels": [
  

## 11 - Read the anatomy of a search response <a class="anchor" id="response-anatomy"></a>
##### [Back to ToC](#TOC)

Typical library-search responses contain:

| Field | Interpretation |
|---|---|
| `q` | Query as represented by the upstream response; useful for checking normalization |
| `total_count` | Upstream count for the search request; see the author-filtering caveat later |
| `page.number` | Current page number |
| `page.num_pages` | Upstream number of pages |
| `results` | Current page of result records |
| `results[].content` | Highlighted snippet fragments, often containing `<em>` markup |
| `results[].passage.urn` | Scaife passage URN for the hit |
| `results[].passage.text` | Indexed text/edition metadata and label hierarchy |
| `author_scope` | Local metadata added only when the MCP `author` option was used |

Search snippets are finding aids. They can contain editorial material, large contexts, repeated highlights, or HTML tags. Preserve the passage URN and retrieve the passage through an appropriate Scaife or CTS tool before making a substantive textual claim.

In [8]:
first_form_result = (form_search.get("results") or [None])[0]

if first_form_result is None:
    print("The live form search returned no result to inspect.")
else:
    print("Top-level response keys:")
    print(sorted(form_search))
    print("\nFirst-result keys:")
    print(sorted(first_form_result))
    print("\nCompact first result:")
    print(json.dumps(compact_search_rows(form_search, 1)[0], ensure_ascii=False, indent=2))
    print("\nRaw first result (clipped for display):")
    print(json.dumps(first_form_result, ensure_ascii=False, indent=2)[:4000])

Top-level response keys:
['kind', 'page', 'page_num', 'q', 'results', 'text_groups', 'total_count', 'type', 'works']

First-result keys:
['content', 'passage']

Compact first result:
{
  "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.75",
  "citation": null,
  "text_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
  "labels": [
    "Homer",
    "Iliad",
    "Ἰλιάς"
  ],
  "snippet": "μῆνιν Ἀπόλλωνος ἑκατηβελέταο ἄνακτος·"
}

Raw first result (clipped for display):
{
  "passage": {
    "url": "/reader/urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.75/",
    "json_url": "/library/passage/urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.75/json/",
    "text_url": "/library/passage/urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.75/text/",
    "text": {
      "url": "/library/urn:cts:greekLit:tlg0012.tlg001.perseus-grc2/",
      "json_url": "/library/urn:cts:greekLit:tlg0012.tlg001.perseus-grc2/json/",
      "text_url": "/library/passage/urn:cts:greekLit:tlg0012.tlg001.pe

## 12 - Compare `instances` and `passages` result formats <a class="anchor" id="result-formats"></a>
##### [Back to ToC](#TOC)

`result_format` is forwarded to Scaife as the library-search `format` parameter:

- `instances` is the default and is suited to inspecting matched instances/snippets;
- `passages` requests a passage-oriented upstream representation.

Do not assume that totals, result grouping, or nested keys are identical across formats. The client should inspect the live response shape and select a format based on the downstream task:

| Task | Starting format |
|---|---|
| Examine highlighted occurrences | `instances` |
| Build a unique passage list | `passages`, followed by explicit URN deduplication |
| Compare formats or debug indexing | request both with identical query/scope arguments |

The following cell uses one scoped phrase query and reports response/result keys instead of relying on a hard-coded shape.

In [9]:
PHRASE_BASE = {
    "query": '"μῆνιν ἄειδε"',
    "language": "greek",
    "query_format": "unicode",
    "search_kind": "form",
    "preserve_operators": True,
    "work": ILIAD_WORK,
}

async with Client(mcp) as client:
    instance_results = await call_json(
        client,
        "search_perseus",
        {**PHRASE_BASE, "result_format": "instances"},
    )
    passage_results = await call_json(
        client,
        "search_perseus",
        {**PHRASE_BASE, "result_format": "passages"},
    )

def response_shape(data):
    first = (data.get("results") or [None])[0]
    return {
        "total_count": data.get("total_count"),
        "returned_results": len(data.get("results") or []),
        "response_keys": sorted(data),
        "first_result_keys": sorted(first) if isinstance(first, dict) else None,
        "first_compact_row": compact_search_rows(data, 1),
    }

print(
    json.dumps(
        {
            "instances": response_shape(instance_results),
            "passages": response_shape(passage_results),
        },
        ensure_ascii=False,
        indent=2,
    )
)

{
  "instances": {
    "total_count": 1,
    "returned_results": 1,
    "response_keys": [
      "kind",
      "page",
      "page_num",
      "q",
      "results",
      "text_groups",
      "total_count",
      "type",
      "works"
    ],
    "first_result_keys": [
      "content",
      "passage"
    ],
    "first_compact_row": [
      {
        "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1",
        "citation": null,
        "text_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
        "labels": [
          "Homer",
          "Iliad",
          "Ἰλιάς"
        ],
        "snippet": "μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος"
      }
    ]
  },
  "passages": {
    "total_count": 1,
    "returned_results": 1,
    "response_keys": [
      "kind",
      "page",
      "page_num",
      "q",
      "results",
      "text_groups",
      "total_count",
      "type",
      "works"
    ],
    "first_result_keys": [
      "content",
      "passage"
    ],
    "first_compact_r

## 13 - Preserve quoted, exclusion, OR, wildcard, and fuzzy syntax <a class="anchor" id="operators"></a>
##### [Back to ToC](#TOC)

Set `preserve_operators=True` when the query intentionally contains syntax that Scaife should interpret. Common observed patterns include:

| Intention | Example expression |
|---|---|
| Quoted phrase | `"μῆνιν ἄειδε"` |
| Exclude a term | `μῆνιν -ἄειδε` |
| OR-style alternatives | `μῆνιν | ἄειδε` |
| Wildcard | `μῆν*` |
| Fuzzy suffix | expression ending in `~` |

Critical implementation detail: `preserve_operators=True` bypasses Greek/Beta Code conversion and returns an NFC-normalized copy of the whole expression. `query_format="betacode"` will therefore **not** convert `mh=nin | a)/eide` while operators are preserved.

Recommended pattern for Greek operator expressions:

```python
{
    "query": '"μῆνιν ἄειδε"',
    "language": "greek",
    "query_format": "unicode",       # documents your intention
    "preserve_operators": True,
}
```

The MCP server preserves the expression; it does not implement or guarantee Scaife's operator semantics. Treat operator behavior as an upstream search feature: scope the query, inspect results, and record the exact expression.

## 14 - Run a compact operator experiment <a class="anchor" id="operator-matrix"></a>
##### [Back to ToC](#TOC)

A controlled operator experiment changes only the query expression while keeping language, search kind, work scope, result format, and page constant.

The cell below runs four expressions inside the *Iliad*. It reports live totals and compact first rows without asserting particular counts. Exact totals are observations, not part of the MCP contract.

If you are exploring an unfamiliar operator, first run a narrow work-scoped query. Broad wildcard or fuzzy searches can produce large result spaces and less interpretable first pages.

In [10]:
operator_queries = {
    "quoted phrase": '"μῆνιν ἄειδε"',
    "exclude ἄειδε": "μῆνιν -ἄειδε",
    "OR-style": "μῆνιν | ἄειδε",
    "wildcard": "μῆν*",
}

operator_results = {}
async with Client(mcp) as client:
    for label, expression in operator_queries.items():
        operator_results[label] = await call_json(
            client,
            "search_perseus",
            {
                "query": expression,
                "language": "greek",
                "query_format": "unicode",
                "search_kind": "form",
                "preserve_operators": True,
                "work": ILIAD_WORK,
                "page_num": 1,
                "result_format": "instances",
            },
        )

operator_report = {
    label: {
        "expression": operator_queries[label],
        "normalized_q": data.get("q"),
        "total_count": data.get("total_count"),
        "first_rows": compact_search_rows(data, 2),
    }
    for label, data in operator_results.items()
}
print(json.dumps(operator_report, ensure_ascii=False, indent=2))

{
  "quoted phrase": {
    "expression": "\"μῆνιν ἄειδε\"",
    "normalized_q": "\"μῆνιν ἄειδε\"",
    "total_count": 1,
    "first_rows": [
      {
        "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1",
        "citation": null,
        "text_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
        "labels": [
          "Homer",
          "Iliad",
          "Ἰλιάς"
        ],
        "snippet": "μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος"
      }
    ]
  },
  "exclude ἄειδε": {
    "expression": "μῆνιν -ἄειδε",
    "normalized_q": "μῆνιν -ἄειδε",
    "total_count": 8,
    "first_rows": [
      {
        "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.75",
        "citation": null,
        "text_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
        "labels": [
          "Homer",
          "Iliad",
          "Ἰλιάς"
        ],
        "snippet": "μῆνιν Ἀπόλλωνος ἑκατηβελέταο ἄνακτος·"
      },
      {
        "passage_urn": "urn:cts:greekLit:tlg001

## 15 - Navigate result pages safely <a class="anchor" id="pagination"></a>
##### [Back to ToC](#TOC)

`page_num` is one-based and must be at least `1`. Scaife determines the page size for library search; `search_perseus` does not expose a library-search page-size argument.

Safe paging pattern:

1. make page 1 request with an explicit query, kind, format, and scope;
2. inspect `page.number` and `page.num_pages`;
3. request later pages with the **same** arguments except `page_num`;
4. retain passage URNs while accumulating results;
5. deduplicate by the identity appropriate to the task;
6. stop at the reported page count or a research-defined limit.

Ranking and the index can change between requests. Pagination is not a permanent snapshot. For reproducible work, record the execution time and search arguments, and avoid assuming that page boundaries remain fixed indefinitely.

In [11]:
PAGED_ARGUMENTS = {
    "query": "καί",
    "language": "greek",
    "query_format": "unicode",
    "search_kind": "form",
    "work": ILIAD_WORK,
    "result_format": "passages",
}

async with Client(mcp) as client:
    page_one = await call_json(
        client,
        "search_perseus",
        {**PAGED_ARGUMENTS, "page_num": 1},
    )
    page_two = await call_json(
        client,
        "search_perseus",
        {**PAGED_ARGUMENTS, "page_num": 2},
    )

page_one_urns = {
    row["passage_urn"] for row in compact_search_rows(page_one, 100) if row["passage_urn"]
}
page_two_urns = {
    row["passage_urn"] for row in compact_search_rows(page_two, 100) if row["passage_urn"]
}

print(
    json.dumps(
        {
            "page_1": summarize_search(page_one, 2),
            "page_2": summarize_search(page_two, 2),
            "passage_urn_overlap": sorted(page_one_urns & page_two_urns),
        },
        ensure_ascii=False,
        indent=2,
    )
)

{
  "page_1": {
    "normalized_q": "καί",
    "total_count": 2694,
    "page_number": 1,
    "num_pages": 270,
    "returned_results": 10,
    "author_scope": null,
    "rows": [
      {
        "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:18.42",
        "citation": null,
        "text_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
        "labels": [
          "Homer",
          "Iliad",
          "Ἰλιάς"
        ],
        "snippet": "καὶ Μελίτη καὶ Ἴαιρα καὶ Ἀμφιθόη καὶ Ἀγαυὴ"
      },
      {
        "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:16.564",
        "citation": null,
        "text_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
        "labels": [
          "Homer",
          "Iliad",
          "Ἰλιάς"
        ],
        "snippet": "Τρῶες καὶ Λύκιοι καὶ Μυρμιδόνες καὶ Ἀχαιοί,"
      }
    ]
  },
  "page_2": {
    "normalized_q": "καί",
    "total_count": 2694,
    "page_number": 2,
    "num_pages": 270,
    "returned_result

## 16 - Use server-side textgroup and work scopes <a class="anchor" id="explicit-scopes"></a>
##### [Back to ToC](#TOC)

`text_group` and `work` are forwarded directly to Scaife, making them the clearest and most efficient library-search scope controls.

| Arguments | Intended boundary |
|---|---|
| neither | Entire indexed library |
| `text_group=...` | One author/textgroup |
| `work=...` | One work across indexed resources |
| both | One work inside the stated textgroup; useful when constructing explicit, auditable requests |

Use work-level rather than edition-level identity in `work`. Search results may identify a Scaife edition such as `perseus-grc2`, while CTS passage retrieval may use another edition such as `perseus-grc1`. Scoping by the shared work URN avoids assuming those editions are interchangeable.

The following comparison uses the same form query at library, Homer textgroup, and *Iliad* work scope. Counts should be interpreted as live upstream observations.

In [12]:
SCOPE_BASE = {
    "query": "μῆνιν",
    "language": "greek",
    "query_format": "unicode",
    "search_kind": "form",
    "result_format": "instances",
}

async with Client(mcp) as client:
    library_scope = await call_json(client, "search_perseus", SCOPE_BASE)
    textgroup_scope = await call_json(
        client,
        "search_perseus",
        {**SCOPE_BASE, "text_group": HOMER_TEXTGROUP},
    )
    work_scope = await call_json(
        client,
        "search_perseus",
        {**SCOPE_BASE, "text_group": HOMER_TEXTGROUP, "work": ILIAD_WORK},
    )

scope_report = {
    "library": summarize_search(library_scope, 1),
    "homer_textgroup": summarize_search(textgroup_scope, 1),
    "iliad_work": summarize_search(work_scope, 1),
}
print(json.dumps(scope_report, ensure_ascii=False, indent=2))

{
  "library": {
    "normalized_q": "μῆνιν",
    "total_count": 314,
    "page_number": 1,
    "num_pages": 32,
    "returned_results": 10,
    "author_scope": null,
    "rows": [
      {
        "passage_urn": "urn:cts:greekLit:tlg2045.tlg001.perseus-grc2:45.238",
        "citation": null,
        "text_urn": "urn:cts:greekLit:tlg2045.tlg001.perseus-grc2",
        "labels": [
          "Nonnus of Panopolis",
          "Dionysiaca",
          "Διονυσιακά"
        ],
        "snippet": "μῆνιν ἀλυσκάζοντες ἀθηήτοιο Λυαίου"
      }
    ]
  },
  "homer_textgroup": {
    "normalized_q": "μῆνιν",
    "total_count": 12,
    "page_number": 1,
    "num_pages": 2,
    "returned_results": 10,
    "author_scope": null,
    "rows": [
      {
        "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.75",
        "citation": null,
        "text_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
        "labels": [
          "Homer",
          "Iliad",
          "Ἰλιάς"
        ],
   

## 17 - Understand author resolution and fallback filtering <a class="anchor" id="author-scope"></a>
##### [Back to ToC](#TOC)

`author` is a convenience layer over the CTS capabilities inventory. Its behavior depends on the resolution result and explicit scopes:

```text
Resolve author against cached CTS capabilities
        |
        +-- exactly one textgroup AND no explicit text_group/work
        |      -> send resolved textgroup to Scaife server-side
        |      -> add author_scope metadata describing that path
        |
        +-- otherwise
               -> make the Scaife request using any explicit scopes
               -> filter only the returned page by matching CTS URN prefixes
               -> add filtered/unfiltered current-page counts
```

A name such as `Homer` may match more than one CTS textgroup depending on the live inventory and matching rules. Passing the discovered textgroup URN is more precise and normally produces the unique server-side path.

Important fallback caveat: local filtering changes `results` for the current page but does not recalculate the upstream `total_count` or page count. In that case, use `author_scope.filtered_page_result_count` to describe the filtered current page; do not report `total_count` as the author's total.

In [13]:
AUTHOR_BASE = {
    "query": '"μῆνιν ἄειδε"',
    "language": "greek",
    "query_format": "unicode",
    "search_kind": "form",
    "preserve_operators": True,
    "result_format": "instances",
}

async with Client(mcp) as client:
    author_by_urn = await call_json(
        client,
        "search_perseus",
        {**AUTHOR_BASE, "author": HOMER_TEXTGROUP},
    )
    author_by_name = await call_json(
        client,
        "search_perseus",
        {**AUTHOR_BASE, "author": "Homer"},
    )

print(
    json.dumps(
        {
            "textgroup_urn_query": summarize_search(author_by_urn, 2),
            "human_name_query": summarize_search(author_by_name, 2),
        },
        ensure_ascii=False,
        indent=2,
    )
)

{
  "textgroup_urn_query": {
    "normalized_q": "\"μῆνιν ἄειδε\"",
    "total_count": 1,
    "page_number": 1,
    "num_pages": 1,
    "returned_results": 1,
    "author_scope": {
      "query": "urn:cts:greekLit:tlg0012",
      "match_count": 1,
      "text_group": "urn:cts:greekLit:tlg0012",
      "note": "Author scope was sent to Scaife as a server-side text_group filter."
    },
    "rows": [
      {
        "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1",
        "citation": null,
        "text_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
        "labels": [
          "Homer",
          "Iliad",
          "Ἰλιάς"
        ],
        "snippet": "μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος"
      }
    ]
  },
  "human_name_query": {
    "normalized_q": "\"μῆνιν ἄειδε\"",
    "total_count": 43,
    "page_number": 1,
    "num_pages": 5,
    "returned_results": 1,
    "author_scope": {
      "query": "Homer",
      "match_count": 2,
      "urns": [
        "urn:cts:gre

## 18 - Query recipes and decision guide <a class="anchor" id="recipes"></a>
##### [Back to ToC](#TOC)

| Research question | Recommended arguments |
|---|---|
| Where does the exact token `μῆνιν` occur? | `query="μῆνιν"`, `query_format="unicode"`, `search_kind="form"` |
| Where do forms indexed under `μῆνις` occur? | `query="μῆνις"`, `search_kind="lemma"` |
| I typed Greek Beta Code | `query="mh=nin"`, `query_format="betacode"` |
| I typed ambiguous ASCII that must stay ASCII | `query_format="unicode"` |
| Find an exact phrase | quote the Unicode phrase and set `preserve_operators=True` |
| Exclude a term | use `term -excluded` with operator preservation |
| Search alternatives | use an observed OR-style expression such as `term1 | term2` and inspect results |
| Search a prefix/wildcard | use the intended `*` expression with operator preservation; scope narrowly first |
| Search only Homer | discover and pass `text_group=HOMER_TEXTGROUP` |
| Search only the *Iliad* | discover and pass `work=ILIAD_WORK` |
| Convenience author search | pass `author`, then inspect `author_scope.note` and match counts |
| Retrieve another page | repeat all arguments and change only `page_num` |
| Build passage-oriented evidence | try `result_format="passages"`, then deduplicate and inspect URNs |
| Search one selected Scaife edition | use `search_within_text`, covered in notebook `08_` |

Practical default for Greek research:

```python
{
    "query": "...",
    "language": "greek",
    "query_format": "unicode",
    "search_kind": "form",
    "page_num": 1,
    "work": DISCOVERED_WORK_URN,
    "result_format": "instances",
}
```

Then change one dimension at a time. Controlled comparisons are easier to interpret than simultaneously changing query spelling, kind, operator syntax, scope, format, and page.

## 19 - Validate search arguments before calling <a class="anchor" id="preflight"></a>
##### [Back to ToC](#TOC)

FastMCP performs authoritative schema validation, and the server validates enums and page bounds. A client-side preflight can catch common mistakes earlier and explain them in research-friendly language.

The helper below checks:

- missing required or unknown arguments against the live schema;
- blank queries;
- invalid enum values;
- nonpositive page numbers;
- likely edition/passage URNs mistakenly passed as `work`;
- operator characters used without preservation;
- Beta Code requested together with operator preservation, where conversion will be bypassed.

This is a convenience check, not a complete JSON Schema validator and not a substitute for server validation.

In [14]:
OPERATOR_CHARACTERS = {'"', "-", "|", "*", "~"}


def preflight_search_arguments(arguments):
    schema = search_tool.inputSchema or {}
    properties = set(schema.get("properties", {}))
    required = set(schema.get("required", []))
    supplied = set(arguments)
    errors = []
    warnings = []

    if missing := sorted(required - supplied):
        errors.append(f"Missing required arguments: {missing}")
    if unknown := sorted(supplied - properties):
        errors.append(f"Unknown arguments: {unknown}")

    query = arguments.get("query")
    if isinstance(query, str) and not query.strip():
        errors.append("query must not be blank")

    if arguments.get("query_format", "auto") not in {"auto", "unicode", "betacode"}:
        errors.append("query_format must be auto, unicode, or betacode")
    if arguments.get("search_kind", "form") not in {"form", "lemma"}:
        errors.append("search_kind must be form or lemma")
    if arguments.get("result_format", "instances") not in {"instances", "passages"}:
        errors.append("result_format must be instances or passages")

    page_num = arguments.get("page_num", 1)
    if not isinstance(page_num, int) or isinstance(page_num, bool) or page_num < 1:
        errors.append("page_num must be an integer of at least 1")

    work = arguments.get("work") or ""
    if work:
        urn_parts = work.split(":")
        is_work_level = (
            len(urn_parts) == 4
            and urn_parts[:2] == ["urn", "cts"]
            and urn_parts[3].count(".") == 1
        )
        if not is_work_level:
            warnings.append("work appears not to be a work-level CTS URN")

    preserve = arguments.get("preserve_operators", False)
    if isinstance(query, str) and any(char in query for char in OPERATOR_CHARACTERS):
        if not preserve:
            warnings.append("query contains operator-like characters but preserve_operators is false")
    if preserve and arguments.get("query_format") == "betacode":
        warnings.append("operator preservation bypasses Beta Code conversion")

    return {
        "ready": not errors,
        "errors": errors,
        "warnings": warnings,
    }


preflight_examples = {
    "valid scoped form search": FORM_ARGUMENTS,
    "operator preservation omitted": {
        "query": "μῆνιν | ἄειδε",
        "query_format": "unicode",
    },
    "Beta Code plus preserved operators": {
        "query": "mh=nin | a)/eide",
        "query_format": "betacode",
        "preserve_operators": True,
    },
    "invalid page": {"query": "μῆνιν", "page_num": 0},
}

print(
    json.dumps(
        {
            label: preflight_search_arguments(arguments)
            for label, arguments in preflight_examples.items()
        },
        ensure_ascii=False,
        indent=2,
    )
)

{
  "valid scoped form search": {
    "ready": true,
    "errors": [],
    "warnings": []
  },
  "operator preservation omitted": {
    "ready": true,
    "errors": [],
    "warnings": [
      "query contains operator-like characters but preserve_operators is false"
    ]
  },
  "Beta Code plus preserved operators": {
    "ready": true,
    "errors": [],
    "warnings": [
      "operator preservation bypasses Beta Code conversion"
    ]
  },
  "invalid page": {
    "ready": false,
    "errors": [
      "page_num must be an integer of at least 1"
    ],
    "warnings": []
  }
}


## 20 - Build an end-to-end evidence packet <a class="anchor" id="workflow"></a>
##### [Back to ToC](#TOC)

A reproducible search artifact should keep more than snippets. At minimum record:

1. the research question;
2. the exact MCP tool name and argument object;
3. the execution time;
4. the discovered textgroup/work scope;
5. normalized query and pagination metadata returned upstream;
6. passage URNs, labels, citations, and snippets selected as evidence;
7. any `author_scope` metadata or limitations.

The packet below is deliberately compact. A larger research workflow can serialize the complete raw response separately and keep a checksum or file reference beside the curated evidence rows.

Search evidence identifies where to investigate. Retrieve and read the cited passage before treating a snippet as sufficient textual evidence.

In [15]:
EVIDENCE_ARGUMENTS = {
    "query": '"μῆνιν ἄειδε"',
    "language": "greek",
    "query_format": "unicode",
    "search_kind": "form",
    "preserve_operators": True,
    "page_num": 1,
    "text_group": HOMER_TEXTGROUP,
    "work": ILIAD_WORK,
    "result_format": "passages",
}

preflight = preflight_search_arguments(EVIDENCE_ARGUMENTS)
if not preflight["ready"]:
    raise ValueError(preflight)

async with Client(mcp) as client:
    evidence_search = await call_json(
        client,
        "search_perseus",
        EVIDENCE_ARGUMENTS,
    )

evidence_packet = {
    "research_question": "Where does the exact opening expression μῆνιν ἄειδε occur in the Iliad search scope?",
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
    "tool": "search_perseus",
    "arguments": EVIDENCE_ARGUMENTS,
    "scope": {
        "text_group": HOMER_TEXTGROUP,
        "work": ILIAD_WORK,
    },
    "response_summary": {
        "normalized_q": evidence_search.get("q"),
        "total_count": evidence_search.get("total_count"),
        "page": evidence_search.get("page"),
        "author_scope": evidence_search.get("author_scope"),
    },
    "evidence": compact_search_rows(evidence_search, 10),
    "preflight_warnings": preflight["warnings"],
    "caution": "Search snippets are finding aids; retrieve cited passages before textual interpretation.",
}

print(json.dumps(evidence_packet, ensure_ascii=False, indent=2))

{
  "research_question": "Where does the exact opening expression μῆνιν ἄειδε occur in the Iliad search scope?",
  "executed_at_utc": "2026-06-18T17:41:11.128835+00:00",
  "tool": "search_perseus",
  "arguments": {
    "query": "\"μῆνιν ἄειδε\"",
    "language": "greek",
    "query_format": "unicode",
    "search_kind": "form",
    "preserve_operators": true,
    "page_num": 1,
    "text_group": "urn:cts:greekLit:tlg0012",
    "work": "urn:cts:greekLit:tlg0012.tlg001",
    "result_format": "passages"
  },
  "scope": {
    "text_group": "urn:cts:greekLit:tlg0012",
    "work": "urn:cts:greekLit:tlg0012.tlg001"
  },
  "response_summary": {
    "normalized_q": "\"μῆνιν ἄειδε\"",
    "total_count": 1,
    "page": {
      "number": 1,
      "start_index": 1,
      "end_index": 1,
      "has_previous": false,
      "has_next": false,
      "num_pages": 1
    },
    "author_scope": null
  },
  "evidence": [
    {
      "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1",
      "c

## 21 - Interpret counts, snippets, and URNs cautiously <a class="anchor" id="interpretation"></a>
##### [Back to ToC](#TOC)

### Counts

- A form count and a lemma count measure different indexed relationships.
- `instances` and `passages` may group results differently.
- Scaife index updates can change totals and ranking.
- with local author fallback filtering, upstream `total_count` is not a recalculated author total.
- a zero count can mean no indexed hit, a normalization mistake, an overly narrow scope, or an upstream issue.

### Snippets

- Snippets may contain `<em>` highlighting, editorial text, or more context than expected.
- The first page is ranked search output, not canonical work order.
- A highlighted string does not by itself establish morphology, syntax, semantics, or textual authenticity.
- Retrieve the passage and inspect surrounding context before analysis.

### URNs

- Keep the complete search-hit passage URN with every extracted row.
- Scaife search editions and Perseus CTS editions can differ even when textgroup, work, and citation match.
- Do not pass a Scaife edition URN blindly to a Perseus CTS passage tool.
- For CTS navigation, discover a CTS edition, preserve the shared work/citation, and document any mapping decision.

### Operators and lemmatization

- Operator syntax is interpreted upstream and should be empirically checked.
- Lemma search reflects the index's morphological analysis, including its omissions and ambiguities.
- Record exact query strings rather than paraphrasing them in research notes.

## 22 - Network, performance, and reproducibility <a class="anchor" id="operations"></a>
##### [Back to ToC](#TOC)

Operational characteristics:

- every `search_perseus` call contacts the live Scaife search service;
- `author` may first load cached Perseus CTS capabilities metadata;
- the first author/discovery operation can therefore be slower and may populate the local cache;
- broad wildcards, common lemmas, and unscoped expressions can produce very large result sets;
- only one library-search page is returned per call;
- repeated loops create repeated upstream requests—set explicit research limits;
- JSON snippets can be large even when only a few results are returned.

For responsible and reproducible work:

- discover and use the narrowest valid scope;
- start with page 1 and a specific query;
- avoid asserting fixed totals in notebooks or tests;
- log tool name, complete arguments, UTC execution time, normalized query, scope, page, format, and URNs;
- save raw responses outside the notebook when an exact historical snapshot matters;
- distinguish live-service observations from stable local tool behavior;
- do not hammer the service with uncontrolled paging, wildcard matrices, or agent loops.

## 23 - Troubleshooting reference <a class="anchor" id="troubleshooting"></a>
##### [Back to ToC](#TOC)

| Symptom | Likely cause and response |
|---|---|
| `fastmcp` cannot be imported | Run the install cell in the active kernel, then restart/rerun if needed |
| `src/perseus_mcp/server.py` cannot be found | Open the notebook within the repository or correct the root path |
| Live schema lacks an argument shown here | Reload `perseus_mcp.server`; if drift remains, update the notebook and server documentation together |
| `logos` unexpectedly becomes Greek | `query_format="auto"` detected short ASCII Beta Code; force `unicode` to preserve ASCII |
| Beta Code operator query is not converted | `preserve_operators=True` intentionally bypasses Beta Code conversion; convert the Greek terms first |
| `*` or `|` behaves strangely | Without preservation these are Beta Code markers; with preservation Scaife interprets them as query syntax |
| Greek decomposed accents behave inconsistently | The server NFC-normalizes outgoing Greek/operator expressions; inspect the returned `q` |
| Search returns other authors | `language` is not a corpus filter; provide `text_group`, `work`, or a precisely resolved `author` |
| Author name gives unexpected scope | Inspect `author_scope.match_count`, `text_group`, `urns`, and `note`; use a discovered textgroup URN for precision |
| Author-filtered results disagree with `total_count` | Ambiguous resolution triggered current-page local filtering; use filtered page counts, not upstream total as an author total |
| `page_num=0` fails | Library pages are one-based and must be at least 1 |
| A work scope yields no result | Verify that it is a work-level URN and that Scaife indexes a resource under that work |
| Result-format parsing fails | Inspect raw response/result keys; do not assume `instances` and `passages` are identical |
| Search hit edition fails in CTS retrieval | Scaife and Perseus CTS edition URNs may differ; discover a CTS edition and map cautiously |
| First author-scoped search is slow | CTS capabilities may be downloading or loading from cache |
| Live request times out or returns an upstream error | Record arguments, retry later, and avoid changing multiple search dimensions while diagnosing |

When reporting a search bug, include the complete argument object, returned normalized `q`, relevant URNs, response/error prefix, execution time, and whether the failure occurred before or after author resolution.

## 24 - Continue learning <a class="anchor" id="next-steps"></a>
##### [Back to ToC](#TOC)

Related notebooks:

- [`04_mcp_greek_search_and_navigation.ipynb`](04_mcp_greek_search_and_navigation.ipynb) — translate search hits into a CTS reading/navigation workflow;
- [`05_mcp_all_tools.ipynb`](05_mcp_all_tools.ipynb) — inspect the full live tool schema catalog;
- [`06_openrouter_llm_mcp_interaction.ipynb`](06_openrouter_llm_mcp_interaction.ipynb) — expose search tools to a constrained LLM tool loop;
- [`08_mcp_cache_and_search_tools.ipynb`](08_mcp_cache_and_search_tools.ipynb) — test cache controls, paged references, reader search, highlights, and Scaife-native retrieval;
- [`09_openrouter_philo_politeia_analysis.ipynb`](09_openrouter_philo_politeia_analysis.ipynb) — build a scoped search evidence packet for a philological synthesis.

Suggested exercises:

1. compare Unicode `μῆνιν` and Beta Code `mh=nin` under identical *Iliad* scope;
2. compare `λόγος` as form and lemma inside one discovered work;
3. test one operator at a time and document live count changes without treating them as permanent;
4. compare `instances` and `passages`, then write a deduplicator appropriate to your question;
5. contrast `author="Homer"` with `author=HOMER_TEXTGROUP` and explain the `author_scope` metadata;
6. page through a scoped query with a fixed maximum and collect unique passage URNs;
7. feed only a reviewed evidence packet—not raw broad search output—to an LLM.

## 25 - Sources <a class="anchor" id="sources"></a>
##### [Back to ToC](#TOC)

This notebook is grounded in:

- the current `search_perseus` implementation and normalization helpers in [`src/perseus_mcp/server.py`](../src/perseus_mcp/server.py);
- the live FastMCP schema returned by `client.list_tools()`;
- project behavior documented in the [README](../README.md), [`docs/architecture.md`](../docs/architecture.md), and [`docs/enduser.md`](../docs/enduser.md);
- local tests in [`tests/test_greek_query_normalization.py`](../tests/test_greek_query_normalization.py) and [`tests/test_exploration_tools.py`](../tests/test_exploration_tools.py);
- [FastMCP](https://github.com/jlowin/fastmcp) for the MCP client/server interface;
- the [Perseus Digital Library](https://www.perseus.tufts.edu/) CTS inventory used for author and work discovery;
- the [Scaife Viewer](https://scaife.perseus.org/) JSON search service used for indexed library search.

Stable local behavior includes argument validation, normalization rules, parameter forwarding, author-resolution logic, and result wrapping. Search counts, ranking, indexed resources, snippets, and operator interpretation are live upstream behavior and can change.

## 26 - Required libraries <a class="anchor" id="required-libraries"></a>
##### [Back to ToC](#TOC)

The repository requires **Python 3.11 or newer**. Recommended installation from the repository root:

```bash
pip install -e .
```

or:

```bash
uv sync
```

Principal third-party libraries used by this notebook:

- `fastmcp>=2.12.0` for the MCP client and local server;
- `httpx>=0.27.0`, used by the server for Scaife and Perseus HTTP requests;
- `python-dotenv>=1.0.0`, installed only for notebook environment-file support and not declared as a core package dependency;
- Jupyter/IPython for asynchronous cells and rendered output.

`datetime`, `html`, `importlib`, `json`, `os`, `pathlib`, `re`, `sys`, and `unicodedata` are Python standard-library modules.

## 27 - Notebook version <a class="anchor" id="notebook-version"></a>
##### [Back to ToC](#TOC)

<div style="float: left;">
  <table>
    <tr>
      <td><strong>Author</strong></td>
      <td>Tony Jurg</td>
    </tr>
    <tr>
      <td><strong>Version</strong></td>
      <td>2.0</td>
    </tr>
    <tr>
      <td><strong>Date</strong></td>
      <td>June 18, 2026</td>
    </tr>
  </table>
</div>